### A Sample Chain
* RAG

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks

In [0]:
llm = ChatDatabricks(
    endpoint = 'databricks-claude-3-7-sonnet',
    extra_params = {"temperature": 0.1}
)

In [0]:
# Create Databricks Vector Search Retrieval
from databricks_langchain.vectorstores import DatabricksVectorSearch
from databricks_langchain.embeddings import DatabricksEmbeddings

emebedding_model = DatabricksEmbeddings(endpoint='databricks-gte-large-en')

# Turn the Vector Search index into a LangChain retriever
vector_search_as_retriever = DatabricksVectorSearch(
    endpoint='one-env-shared-endpoint-14',
    index_name='sandbox_db.val_dataset_redshift.redshift_pdf_documentation_managed_vs_index',
    text_column='content',
    embedding=emebedding_model, 
    columns=[
        'id',
        'content',
        'url',
    ],
).as_retriever(search_kwargs={"k": 3, "query_type": "ann"})

In [0]:
type(vector_search_as_retriever)

In [0]:
docs = vector_search_as_retriever.invoke('What is LISTAGG function in Redshift?')

In [0]:
docs[0]

In [0]:
type(docs[0])

In [0]:
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStoreRetriever

In [0]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [0]:
#langChain prompt Template
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

#langChain plain string output parser {wrapper on top of Pydantic} - These modules are for formatting LLM output
output_parser = StrOutputParser()

In [0]:
# create langchain chain
from langchain_core.runnables import RunnableMap

chain = RunnableMap({
  'context': lambda x: vector_search_as_retriever.get_relevant_documents(f"{x['question']} in the context of {x['topic']}?"),
  'question': lambda x: x['question']  
}) | prompt | llm | output_parser

In [0]:
chain.invoke({'question': 'What is ACTIVITYCOUNT', 'topic': 'Teradata BTEQ'})

### Binding Parameters to LLM Call
* function binding

In [0]:
functions = [
    {
      "name": "weather_search",
      "description": "Search for weather given an airport code",
      "parameters": {
        "type": "object",
        "properties": {
          "airport_code": {
            "type": "string",
            "description": "The airport code to get the weather for"
          },
        },
        "required": ["airport_code"]
      }
    },
        {
      "name": "sports_search",
      "description": "Search for news of recent sport events",
      "parameters": {
        "type": "object",
        "properties": {
          "team_name": {
            "type": "string",
            "description": "The sports team to search for"
          },
        },
        "required": ["team_name"]
      }
    }
  ]

In [0]:
messages = [
  ("human", "{input}")
]
prompt = ChatPromptTemplate.from_messages(messages)
model = ChatDatabricks(
    endpoint = 'my-gpt-endpoint',
    extra_params = {"temperature": 0}
).bind(functions=functions)

In [0]:
runnable = prompt | model

In [0]:
runnable.invoke({"input": "What's the weather in SF?"})

In [0]:
runnable.invoke({"input": "how did the patriots do yesterday?"})

### Fallbacks
* in case a runnable fails, we can run a fallback runnable

In [0]:
from langchain.llms import OpenAI
import json

In [0]:
model = ChatDatabricks(
    endpoint = 'databricks-claude-3-7-sonnet',
    extra_params = {"temperature": 0.1}
)

simple_chain = model | json.loads



In [0]:
challenge = "write three poems in a json blob, where each poem is a json blob of a title, author, and first line"
simple_chain.invoke(challenge)
# This is supposed to fail

In [0]:
chain = model | StrOutputParser() 
final_chain = simple_chain.with_fallbacks([chain])
final_chain.invoke(challenge)

### Interface

In [0]:
prompt = ChatPromptTemplate.from_template(
    "Tell me a short joke about {topic}"
)
model =  ChatDatabricks(
    endpoint = 'databricks-claude-3-7-sonnet',
    extra_params = {"temperature": 0.1}
)

output_parser = StrOutputParser()

chain = prompt | model | output_parser

In [0]:
chain.invoke({"topic": "bears"})

In [0]:
chain.batch([{"topic": "bears"}, {"topic": "frogs"}])

In [0]:
for t in chain.stream({"topic": "bears"}):
    print(t)

In [0]:
response = await chain.ainvoke({"topic": "bears"})
response